In [1]:
import pandas as pd
from io import StringIO
from urllib.request import Request, urlopen

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
}
request = Request(url, headers=headers)
html = urlopen(request).read().decode('utf-8')

tables = pd.read_html(StringIO(html), attrs={'id': 'constituents'})
table = tables[0]
tickers = table['Symbol'].tolist()
print(len(tickers))

503


In [2]:
import os
from pathlib import Path
import yfinance as yf

output_dir = Path('data') / 'tickers'
output_dir.mkdir(parents=True, exist_ok=True)

for i, ticker in enumerate(tickers, start=1):
    safe_ticker = ticker.replace('/', '-').replace('.', '-')
    df = yf.download(ticker, period='2y', interval='1d', progress=False)
    if df.empty:
        print(f"[{i}/{len(tickers)}] {ticker}: no data")
        continue
    df.to_csv(output_dir / f"{safe_ticker}.csv")
    if i % 25 == 0 or i == len(tickers):
        print(f"Saved {i}/{len(tickers)} tickers")

Saved 25/503 tickers
Saved 50/503 tickers


$BRK.B: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['BRK.B']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


[62/503] BRK.B: no data
Saved 75/503 tickers


$BF.B: possibly delisted; no price data found  (period=2y)

1 Failed download:
['BF.B']: possibly delisted; no price data found  (period=2y)


[77/503] BF.B: no data
Saved 100/503 tickers
Saved 125/503 tickers
Saved 150/503 tickers
Saved 175/503 tickers
Saved 200/503 tickers
Saved 225/503 tickers
Saved 250/503 tickers
Saved 275/503 tickers
Saved 300/503 tickers
Saved 325/503 tickers
Saved 350/503 tickers
Saved 375/503 tickers
Saved 400/503 tickers
Saved 425/503 tickers
Saved 450/503 tickers
Saved 475/503 tickers
Saved 500/503 tickers
Saved 503/503 tickers


In [3]:
exclude_positions = {62, 77}
filtered = [t for idx, t in enumerate(tickers, start=1) if idx not in exclude_positions]

out_path = Path('data') / 'tickers.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'ticker': filtered}).to_csv(out_path, index=False)
print(f"Saved {len(filtered)} tickers to {out_path}")

Saved 501 tickers to data\tickers.csv


In [3]:
from pathlib import Path
import pandas as pd
data_dir = Path('data') / 'tickers'
csv_paths = sorted(data_dir.glob('*.csv'))

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")

for path in csv_paths:
    df = pd.read_csv(path)
    if df.empty:
        continue
    # Remove the first data row after headers
    df = df.iloc[1:].copy()
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.sort_values('Date')
    df = df.drop_duplicates()
    if 'Close' in df.columns:
        df = df.dropna(subset=['Close'])
    df = df.reset_index(drop=True)
    df.to_csv(path, index=False)

print(f"Processed {len(csv_paths)} files in {data_dir}")

Processed 501 files in data\tickers
